# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library, following the Croissant schema. All dataset entities—such as record sets, fields, and columns—are referenced via their `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset is provided via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's examine available record sets, fields, and columns with their `@id` fields, as required by the Croissant schema.

In [ ]:
# List all record sets and their fields by `@id`

record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")

record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print('-' * 40)

## 3. Data Extraction

For further exploration, we'll load the data from each record set using each set's `@id`. You can look up the fields you want to focus on by their `@id` as listed above.

In [ ]:
# Load all data into pandas DataFrames, key'd by record set `@id`
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set '{record_set_id}'. Columns (fields):")
        print(df.columns.tolist())
        print('-'*30)

# For demonstration, select the first available record set (update if you want another)
if dataframes:
    selected_record_set = list(dataframes.keys())[0]
    print(f"Using record set `@id`: {selected_record_set}")
    display(dataframes[selected_record_set].head(5))

## 4. Exploratory Data Analysis (EDA)

We'll perform basic EDA steps:
- Filter numeric records based on a threshold
- Normalize a numeric field
- Group by a categorical field

For demonstration, we'll try to use typical numeric and categorical fields by their `@id`. Check above for available field `@id`s.

In [ ]:
# -- EDA Section --

# Update these to match appropriate field ids from your specific dataset structure:
# For demonstration, try to automatically select the first numeric and groupable field
df = dataframes[selected_record_set]
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
group_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < max(10, len(df)//5)]

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Selected numeric field for EDA: {numeric_field_id}")
    threshold = df[numeric_field_id].quantile(0.3)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (Total: {len(filtered_df)})")
    display(filtered_df.head(5))

    # Normalize
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / (std if std != 0 else 1)
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(5))
else:
    print('No numeric field detected for EDA.')

if group_fields and numeric_fields:
    group_field_id = group_fields[0]
    print(f"\nGrouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean for {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head(10))
else:
    print('No suitable group field detected for grouping.')

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized values, and show group comparisons (if grouping fields exist).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(12, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True, label="Original", color="skyblue")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, color="orange", label="Normalized")
    plt.title(f"Distribution of Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.legend()
    plt.show()

if group_fields and numeric_fields:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, palette="viridis")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, you learned how to:
- Load clinical and molecular colorectal cancer records via Croissant schema using `mlcroissant`
- Reference all record sets and fields by their `@id` as per FAIR best practices
- Extract records into pandas DataFrames and perform basic EDA
- Normalize, filter, and group data for further clinical or statistical analysis
- Visualize field distributions and group statistics

You can now extend this template for further analysis—refer to entity `@id` keys to ensure robust, reproducible workflows!